## Setting up Configuration

In [0]:
%run ./00_setup_config

## Reading data from weather_curated table

In [0]:
df_curated = spark.table("internship_databricks_ws.default.weather_curated")

In [0]:
from pyspark.sql.functions import (
    to_date, dayofmonth, date_format, month, quarter, year
)

# ---------------------------------------------------------
# 1. Extract distinct dates from curated data
# ---------------------------------------------------------
df_dates = df_curated.select(to_date("weather_time_pkt").alias("date")).distinct()

# ---------------------------------------------------------
# 2. Build full attribute set + deterministic date_id
#    (yyyyMMdd -> e.g. 2026-08-17 becomes 20260817)
# ---------------------------------------------------------
df_dims_date_new = (
    df_dates
    .withColumn("date_id", date_format("date", "yyyyMMdd").cast("int"))
    .withColumn("day", dayofmonth("date"))
    .withColumn("day_name", date_format("date", "EEEE"))
    .withColumn("month", month("date"))
    .withColumn("month_name", date_format("date", "MMMM"))
    .withColumn("quarter", quarter("date"))
    .withColumn("year", year("date"))
    .select("date_id", "date", "day", "day_name", "month", "month_name", "quarter", "year")
)

# ---------------------------------------------------------
# 3. Create table once if it doesn't exist yet
# ---------------------------------------------------------
spark.sql(f"""
CREATE TABLE IF NOT EXISTS internship_databricks_ws.default.weather_dims_date
USING DELTA
LOCATION '{gold_path}weather_dims_date/'
""")

# ---------------------------------------------------------
# 4. Append only dates not already in the dim
# ---------------------------------------------------------
df_dims_date_existing = spark.table("internship_databricks_ws.default.weather_dims_date")
new_dates_to_append = df_dims_date_new.join(
    df_dims_date_existing.select("date_id"), on="date_id", how="left_anti"
)

if new_dates_to_append.count() > 0:
    new_dates_to_append.write.format("delta").mode("append") \
        .saveAsTable("internship_databricks_ws.default.weather_dims_date")

# ---------------------------------------------------------
# 5. Read final dim
# ---------------------------------------------------------
df_dims_date = spark.table("internship_databricks_ws.default.weather_dims_date")
df_dims_date.orderBy("date_id").show(50, truncate=False)

In [0]:
from pyspark.sql.functions import to_timestamp, hour, minute, col, when

df_curated = spark.table("internship_databricks_ws.default.weather_curated")

# 1. Parse timestamp, then extract unique hour/minute combinations
df_times = (
    df_curated
    .withColumn("weather_timestamp", to_timestamp("weather_time_pkt", "yyyy-MM-dd'T'HH:mm"))
    .select(
        hour("weather_timestamp").alias("hour"),
        minute("weather_timestamp").alias("minute")
    )
    .distinct()
)

# 2. Build deterministic time_id (e.g. 09:15 -> 915, 12:30 -> 1230) + time_period
df_dims_time_new = (
    df_times
    .withColumn("time_id", (col("hour") * 100 + col("minute")).cast("int"))
    .withColumn(
        "time_period",
        when((col("hour") >= 0) & (col("hour") < 6), "Night")
        .when((col("hour") >= 6) & (col("hour") < 12), "Morning")
        .when((col("hour") >= 12) & (col("hour") < 18), "Afternoon")
        .otherwise("Evening")
    )
    .select("time_id", "hour", "minute", "time_period")
)

# 3. Create table once if it doesn't exist
spark.sql(f"""
CREATE TABLE IF NOT EXISTS internship_databricks_ws.default.weather_dims_time
USING DELTA
LOCATION '{gold_path}weather_dims_time/'
""")

# 4. Append only time combinations not already in the dim
df_dims_time_existing = spark.table("internship_databricks_ws.default.weather_dims_time")
new_times_to_append = df_dims_time_new.join(
    df_dims_time_existing.select("time_id"), on="time_id", how="left_anti"
)

if new_times_to_append.count() > 0:
    new_times_to_append.write.format("delta").mode("append") \
        .saveAsTable("internship_databricks_ws.default.weather_dims_time")

# 5. Read final dim
df_dims_time = spark.table("internship_databricks_ws.default.weather_dims_time")
df_dims_time.orderBy("time_id").show(100, truncate=False)

In [0]:
## Creating weather_dims_condition

from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType
)

weather_conditions = [
    (0, "Clear sky", "Clear"),
    (1, "Mainly clear", "Clear"),
    (2, "Partly cloudy", "Cloudy"),
    (3, "Overcast", "Cloudy"),
    (45, "Fog", "Fog"),
    (48, "Depositing rime fog", "Fog"),
    (51, "Light drizzle", "Drizzle"),
    (53, "Moderate drizzle", "Drizzle"),
    (55, "Dense drizzle", "Drizzle"),
    (56, "Light freezing drizzle", "Freezing Drizzle"),
    (57, "Dense freezing drizzle", "Freezing Drizzle"),
    (61, "Slight rain", "Rain"),
    (63, "Moderate rain", "Rain"),
    (65, "Heavy rain", "Rain"),
    (66, "Light freezing rain", "Freezing Rain"),
    (67, "Heavy freezing rain", "Freezing Rain"),
    (71, "Slight snow fall", "Snow"),
    (73, "Moderate snow fall", "Snow"),
    (75, "Heavy snow fall", "Snow"),
    (77, "Snow grains", "Snow"),
    (80, "Slight rain showers", "Rain"),
    (81, "Moderate rain showers", "Rain"),
    (82, "Violent rain showers", "Rain"),
    (85, "Slight snow showers", "Snow"),
    (86, "Heavy snow showers", "Snow"),
    (95, "Thunderstorm", "Storm"),
    (96, "Thunderstorm with slight hail", "Storm"),
    (99, "Thunderstorm with heavy hail", "Storm")
]

condition_schema = StructType([
    StructField("weather_code", IntegerType(), False),
    StructField("condition", StringType(), False),
    StructField("category", StringType(), False)
])

df_dims_condition = spark.createDataFrame(
    weather_conditions,
    schema=condition_schema
)

df_dims_condition.show(truncate=False)

In [0]:
## Creating dimension tables in Gold layer

# Date dimension
spark.sql(f"""
CREATE TABLE IF NOT EXISTS internship_databricks_ws.default.weather_dims_date
USING DELTA
LOCATION '{gold_path}weather_dims_date/'
""")

df_dims_date.write \
    .format("delta") \
    .mode("overwrite") .option("mergeSchema", "true") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "internship_databricks_ws.default.weather_dims_date"
    )


# Time dimension
spark.sql(f"""
CREATE TABLE IF NOT EXISTS internship_databricks_ws.default.weather_dims_time
USING DELTA
LOCATION '{gold_path}weather_dims_time/'
""")

df_dims_time.write \
    .format("delta") \
    .mode("overwrite") .option("mergeSchema", "true") \
    .saveAsTable(
        "internship_databricks_ws.default.weather_dims_time"
    )


# Weather condition dimension
spark.sql(f"""
CREATE TABLE IF NOT EXISTS internship_databricks_ws.default.weather_dims_condition
USING DELTA
LOCATION '{gold_path}weather_dims_condition/'
""")

df_dims_condition.write \
    .format("delta") \
    .mode("overwrite") .option("mergeSchema", "true") \
    .saveAsTable(
        "internship_databricks_ws.default.weather_dims_condition"
    )

print("All three dimension tables written to Gold layer.")

## Splitting weather_curated + adding unique id

In [0]:
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window
from pyspark.sql.functions import max as spark_max


df_dims_city = spark.table("internship_databricks_ws.default.weather_dims_city").select("city", "city_id")
df_cities = df_curated.select("city").distinct()

new_cities = df_cities.join(df_dims_city, on="city", how="left_anti")
max_id = df_dims_city.agg(spark_max("city_id")).collect()[0][0] or 0
if new_cities.count() > 0:
    new_cities = new_cities.withColumn("city_id", row_number().over(Window.orderBy("city")) + max_id).select("city_id", "city")
    new_cities.write.format("delta").mode("append").saveAsTable("internship_databricks_ws.default.weather_dims_city")
df_dims_city = spark.table("internship_databricks_ws.default.weather_dims_city")
df_dims_city.show(truncate=False)

## Joining weather_curated with city_id,date_id,time_id,weather_timestamp by matching cities

In [0]:
from pyspark.sql.functions import to_timestamp, date_format, hour, minute, col

# ---------------------------------------------------------
# 1. Parse timestamp once, derive keys directly (no dim joins needed
#    for date_id / time_id since they're deterministic formulas)
# ---------------------------------------------------------
df_fact_source = (
    df_curated
    .withColumn("weather_timestamp", to_timestamp("weather_time_pkt", "yyyy-MM-dd'T'HH:mm"))
    .withColumn("date_id", date_format("weather_timestamp", "yyyyMMdd").cast("int"))
    .withColumn("time_id", (hour("weather_timestamp") * 100 + minute("weather_timestamp")).cast("int"))
)

# ---------------------------------------------------------
# 2. Join only city dimension (city_id is a surrogate key, not derivable)
# ---------------------------------------------------------
df_fact_new = (
    df_fact_source
    .join(df_dims_city.select("city", "city_id"), on="city", how="left")
    .select(
        "city_id", "date_id", "time_id", "weather_time_pkt",
        "temperature_c", "humidity_pct", "feels_like_c", "precipitation_mm",
        "weather_code", "pressure_hpa", "windspeed_kmh", "winddirection_deg",
        "cloud_cover_pct", "is_day", "loading_time"
    )
)

df_fact_new.show(20, truncate=False)

## Creating weather_dims_city table + writing df_dims_city's data in it

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS internship_databricks_ws.default.weather_dims_city
USING DELTA
LOCATION '{gold_path}weather_dims_city/'
""")

print("Written to gold layer")

## Dedup Logic

In [0]:
try:
    df_fact_existing = spark.table("internship_databricks_ws.default.weather_fact_weather")
    df_fact_to_append = df_fact_new.join(
        df_fact_existing.select("city_id", "date_id", "time_id"),
        on=["city_id", "date_id", "time_id"], how="left_anti"
    )
except Exception:
    df_fact_to_append = df_fact_new

df_fact_to_append.write.format("delta").mode("append") \
    .saveAsTable("internship_databricks_ws.default.weather_fact_weather")

## Creating weather_fact_weather table + writing df_fact_weather's data in it

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS internship_databricks_ws.default.weather_fact_weather
USING DELTA
LOCATION '{gold_path}weather_fact_weather/'
""")
df_fact_to_append.write.format("delta").mode("append").saveAsTable("internship_databricks_ws.default.weather_fact_weather")
print("Appended new readings to fact_weather")

## Aggregating weather_summary for cities

In [0]:
%sql
CREATE OR REPLACE TABLE internship_databricks_ws.default.weather_summary
USING DELTA
LOCATION 'abfss://gold@internshipdlsa01.dfs.core.windows.net/weather_summary/'
AS
SELECT
    d.city,
    ROUND(AVG(f.temperature_c), 1) AS avg_temp_c,
    ROUND(MAX(f.temperature_c), 1) AS max_temp_c,
    ROUND(MIN(f.temperature_c), 1) AS min_temp_c,
    ROUND(AVG(f.humidity_pct), 1) AS avg_humidity_pct,
    ROUND(AVG(f.windspeed_kmh), 1) AS avg_windspeed_kmh,
    COUNT(*) AS reading_count
FROM internship_databricks_ws.default.weather_fact_weather f
JOIN internship_databricks_ws.default.weather_dims_city d ON f.city_id = d.city_id
GROUP BY d.city
ORDER BY d.city

## Finding rainy cities

In [0]:
%sql
CREATE OR REPLACE TABLE internship_databricks_ws.default.weather_rainy_cities
USING DELTA
LOCATION 'abfss://gold@internshipdlsa01.dfs.core.windows.net/weather_rainy_cities/'
AS
SELECT c.city, f.precipitation_mm, f.humidity_pct
FROM internship_databricks_ws.default.weather_dims_city c join 
weather_fact_weather f on c.city_id = f.city_id
WHERE precipitation_mm > 0

## Calculating temperature range for cities

In [0]:
%sql
Create or replace table internship_databricks_ws.default.weather_cities_range
using delta location
'abfss://gold@internshipdlsa01.dfs.core.windows.net/weather_cities_range/'
as
Select c.city, avg(f.temperature_c) as avg_temperature_c, ROUND(max(f.temperature_c) - min(f.temperature_c),2) as temp_range from internship_databricks_ws.default.weather_dims_city c join weather_fact_weather f on c.city_id = f.city_id
group by c.city
order by temp_range

In [0]:
display(
    df_curated
    .select("weather_time_pkt")
    .distinct()
    .orderBy("weather_time_pkt")
    .limit(20)
)

In [0]:
df_curated.printSchema()

In [0]:
df_fact = spark.table("internship_databricks_ws.default.weather_fact_weather")

# Confirm new columns exist
df_fact.printSchema()

# Confirm no duplicate keys
df_fact.groupBy("city_id", "date_id", "time_id").count().filter("count > 1").show()

# Confirm no null FKs (would indicate a join issue somewhere)
df_fact.filter("city_id IS NULL OR date_id IS NULL OR time_id IS NULL").count()